# 03 - Text Translation

## Purpose
Classifies each parsed document page as `ENGLISH`, `BILINGUAL`, or `NON_ENGLISH`
and translates pages that require it before field extraction runs. This ensures
the extraction model always receives English text regardless of the source
document language.

## What this notebook does

### Language Classification
Each page in `DOCUMENTS_PAGES` is passed through a two-pass AI classifier:

**Pass 1 — Haiku (binary screen)** runs first using `claude-haiku-4-5` in
parallel SQL. It asks a simple binary question — is this page purely English
or does it contain any non-English content? Pages classified as `ENGLISH` by
Haiku are written directly — no further AI call needed. Pages where Haiku
detects any non-English content are passed to Sonnet.

**Pass 2 — Sonnet (BILINGUAL vs NON_ENGLISH)** runs only on pages Haiku
flagged as containing non-English content. Sonnet determines whether the page
is `BILINGUAL` (every non-English value has an English equivalent on the same
page — no translation needed) or `NON_ENGLISH` (field values in non-English
with no English equivalent — translation required).

Both prompts are loaded from `PIPELINE_CONFIG` at runtime.

**Classification labels and downstream behavior:**

| Label | Meaning | Action |
|---|---|---|
| `ENGLISH` | All field values in English or language-neutral | No translation |
| `BILINGUAL` | Every non-English value has English equivalent on same page | No translation |
| `NON_ENGLISH` | Field values in non-English with no English equivalent | Translated before extraction |

### Translation
Only `NON_ENGLISH` pages are translated via `AI_COMPLETE` with `claude-sonnet-5`
using parallel SQL. Translation model and prompt loaded from `PIPELINE_CONFIG`.

## Outputs
| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_PAGES` | `LANGUAGE_RESULT`, `LANGUAGE_CLASSIFICATION_REASON`, `LANGUAGE_CHECKED_AT` updated per page |
| `PROCESSING.DOCUMENTS_PAGES` | `PAGE_CONTENT_TRANSLATED` written for `NON_ENGLISH` pages only |
| `AUDIT.LLM_USAGE` | Token consumption per page for Haiku, Sonnet, and translation steps |

## Key design decisions
- **Two AI passes instead of rule-based pre-filter** — rule-based character
  counting was removed; Haiku handles the cheap binary screen reliably and
  consistently across all script types
- **Haiku for binary screen, Sonnet for nuanced classification** — Haiku is
  sufficient and cheap for ENGLISH vs non-ENGLISH; Sonnet handles the harder
  BILINGUAL vs NON_ENGLISH distinction which requires document reasoning
- **Prompts loaded from PIPELINE_CONFIG** — model names and prompt templates
  are config-driven; no notebook edits needed to tune prompts or swap models

In [ ]:
import json
import re
import pandas as pd
import pytz
import uuid
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
PROCESSING_SCHEMA = 'PROCESSING'
AUDIT_SCHEMA = 'AUDIT'
CONFIG_SCHEMA     = 'CONFIG'
s = get_active_session() 

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

def now_ast(): # could be replace with any time zone later
    tz = pytz.timezone('America/Halifax')
    return datetime.now(tz).isoformat()

def parse_json_response(raw):
    """Parse JSON from AI_COMPLETE — handles double-encoding and fences."""
    cleaned = raw.strip()
    if cleaned.startswith('"'):
        cleaned = json.loads(cleaned)
    cleaned = cleaned.strip()
    if '```' in cleaned:
        parts   = cleaned.split('```')
        cleaned = parts[1].strip()
        if cleaned.lower().startswith('json'):
            cleaned = cleaned[4:].strip()
    parsed = json.loads(cleaned)
    if isinstance(parsed, str):
        parsed = json.loads(parsed)
    if not isinstance(parsed, dict):
        raise ValueError(f"Expected dict, got {type(parsed)}")
    return parsed

In [ ]:
#Load prompts from Pipeline Config
config = {
    row['CONFIG_KEY']: row['CONFIG_VALUE']
    for row in s.sql(f"""
        SELECT CONFIG_KEY, CONFIG_VALUE
        FROM {DB}.{CONFIG_SCHEMA}.PIPELINE_CONFIG
        WHERE CONFIG_KEY IN (
            'lang_check_haiku_model',
            'lang_check_haiku_prompt',
            'lang_check_sonnet_model',
            'lang_check_sonnet_prompt',
            'translate_model',
            'translate_prompt'
        )
        AND IS_ACTIVE = TRUE
    """).collect()
}

HAIKU_MODEL      = config.get('lang_check_haiku_model')
HAIKU_PROMPT     = config.get('lang_check_haiku_prompt')
SONNET_MODEL     = config.get('lang_check_sonnet_model')
SONNET_PROMPT    = config.get('lang_check_sonnet_prompt')
TRANSLATE_MODEL  = config.get('translate_model')
TRANSLATE_PROMPT = config.get('translate_prompt')

missing = [k for k, v in {
    'lang_check_haiku_model':   HAIKU_MODEL,
    'lang_check_haiku_prompt':  HAIKU_PROMPT,
    'lang_check_sonnet_model':  SONNET_MODEL,
    'lang_check_sonnet_prompt': SONNET_PROMPT,
    'translate_model':          TRANSLATE_MODEL,
    'translate_prompt':         TRANSLATE_PROMPT,
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing config keys in PIPELINE_CONFIG: {missing} — "
        f"run 00_setup_config.ipynb first"
    )

info(f"Haiku model      : {HAIKU_MODEL}")
info(f"Haiku prompt     : {len(HAIKU_PROMPT)} chars")
info(f"Sonnet model     : {SONNET_MODEL}")
info(f"Sonnet prompt    : {len(SONNET_PROMPT)} chars")
info(f"Translate model  : {TRANSLATE_MODEL}")
info(f"Translate prompt : {len(TRANSLATE_PROMPT)} chars")

In [ ]:
# Load Pages need language check
rows = s.sql(f"""
    SELECT
        DOC_ID,
        PAGE_INDEX,
        PAGE_NUMBER,
        PAGE_CONTENT
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE PAGE_CONTENT      IS NOT NULL
      AND TRIM(PAGE_CONTENT) <> ''
      AND LANGUAGE_RESULT   IS NULL
""").collect()

info(f"Loaded {len(rows)} page(s) for language classification")

if not rows:
    print("\nNothing to classify.")

In [ ]:
pure_english      = []
sonnet_classified = []
ai_classified     = []

if rows:
    values_list = ', '.join(
        f"('{r['DOC_ID']}', {r['PAGE_INDEX']})"
        for r in rows
    )

    # Pass 1 - binary check
    info(f"Pass 1: {HAIKU_MODEL} screening {len(rows)} pages...")

    haiku_results = s.sql(f"""
        SELECT
            p.DOC_ID,
            p.PAGE_INDEX,
            p.PAGE_NUMBER,
            p.PAGE_CONTENT,
            LENGTH(p.PAGE_CONTENT) AS CONTENT_CHARS,
            AI_COMPLETE(
                '{HAIKU_MODEL}',
                REPLACE(?, '{{page_text}}', p.PAGE_CONTENT)
            ) AS HAIKU_RESULT
        FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
        JOIN (
            SELECT DISTINCT DOC_ID, PAGE_INDEX
            FROM (VALUES {values_list}) AS t(DOC_ID, PAGE_INDEX)
        ) AS to_classify
            ON  p.DOC_ID     = to_classify.DOC_ID
            AND p.PAGE_INDEX = to_classify.PAGE_INDEX
        WHERE p.PAGE_CONTENT IS NOT NULL
          AND TRIM(p.PAGE_CONTENT) <> ''
    """, params=[HAIKU_PROMPT]).collect()

    needs_sonnet = []

    for row in haiku_results:
        doc_id     = row['DOC_ID']
        page_index = row['PAGE_INDEX']
        raw        = row['HAIKU_RESULT']

        try:
            parsed = parse_json_response(raw)
            label  = parsed.get('label', 'NON_ENGLISH').upper().strip()
            reason = parsed.get('reason', '')

            if label == 'ENGLISH':
                pure_english.append({
                    'DOC_ID':                         doc_id,
                    'PAGE_INDEX':                     page_index,
                    'PAGE_NUMBER':                    row['PAGE_NUMBER'],
                    'PAGE_CONTENT':                   row['PAGE_CONTENT'],
                    'LANGUAGE_RESULT':                'ENGLISH',
                    'LANGUAGE_CLASSIFICATION_REASON': f'Haiku: {reason}',
                    'ESTIMATED_INPUT_TOKENS':         row['CONTENT_CHARS'] // 4,
                })
            else:
                needs_sonnet.append(row)

        except Exception as e:
            warning(f"  Haiku parse failed for {doc_id} PAGE {page_index}: {e} "
                    f"— sending to Sonnet")
            needs_sonnet.append(row)

    info(f"  Pass 1 complete — ENGLISH: {len(pure_english)} | "
         f"needs Sonnet: {len(needs_sonnet)}")

    # Pass 2 - billingual vs non English
    if needs_sonnet:
        sonnet_values = ', '.join(
            f"('{r['DOC_ID']}', {r['PAGE_INDEX']})"
            for r in needs_sonnet
        )

        info(f"Pass 2: {SONNET_MODEL} classifying {len(needs_sonnet)} pages...")

        sonnet_results = s.sql(f"""
            SELECT
                p.DOC_ID,
                p.PAGE_INDEX,
                p.PAGE_NUMBER,
                p.PAGE_CONTENT,
                LENGTH(p.PAGE_CONTENT) AS CONTENT_CHARS,
                AI_COMPLETE(
                    '{SONNET_MODEL}',
                    REPLACE(?, '{{page_text}}', p.PAGE_CONTENT)
                ) AS SONNET_RESULT
            FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
            JOIN (
                SELECT DISTINCT DOC_ID, PAGE_INDEX
                FROM (VALUES {sonnet_values}) AS t(DOC_ID, PAGE_INDEX)
            ) AS to_classify
                ON  p.DOC_ID     = to_classify.DOC_ID
                AND p.PAGE_INDEX = to_classify.PAGE_INDEX
            WHERE p.PAGE_CONTENT IS NOT NULL
              AND TRIM(p.PAGE_CONTENT) <> ''
        """, params=[SONNET_PROMPT]).collect()

        for row in sonnet_results:
            doc_id     = row['DOC_ID']
            page_index = row['PAGE_INDEX']
            raw        = row['SONNET_RESULT']

            try:
                parsed = parse_json_response(raw)
                label  = parsed.get('label', '').upper().strip()
                reason = parsed.get('reason', '')

                if label not in ('ENGLISH', 'BILINGUAL', 'NON_ENGLISH'):
                    raise ValueError(f"Unexpected label: {label}")

                sonnet_classified.append({
                    'DOC_ID':                         doc_id,
                    'PAGE_INDEX':                     page_index,
                    'PAGE_NUMBER':                    row['PAGE_NUMBER'],
                    'PAGE_CONTENT':                   row['PAGE_CONTENT'],
                    'LANGUAGE_RESULT':                label,
                    'LANGUAGE_CLASSIFICATION_REASON': f'Sonnet: {reason}',
                    'ESTIMATED_INPUT_TOKENS':         row['CONTENT_CHARS'] // 4,
                })

                info(f"  [{label}] {doc_id} PAGE {page_index} — {reason}")

            except Exception as e:
                error(f"  [FAIL] {doc_id} PAGE {page_index}: {e}")
                error(f"  RAW: {str(raw)[:300]}")

    ai_classified = pure_english + sonnet_classified

    info(f"\nClassification complete:")
    info(f"  {HAIKU_MODEL} ENGLISH : {len(pure_english)}")
    info(f"  {SONNET_MODEL}        : {len(sonnet_classified)}")
    info(f"  Total                 : {len(ai_classified)}")

In [ ]:
# UPDATE DOCUMENTS_PAGES & INSERT LLM_USAGE 
language_check = ai_classified

if language_check:
    s.write_pandas(
        pd.DataFrame([{
            'DOC_ID':              row['DOC_ID'],
            'PAGE_INDEX':          row['PAGE_INDEX'],
            'LANGUAGE_RESULT':     row['LANGUAGE_RESULT'],
            'LANGUAGE_CLASSIFICATION_REASON':  row.get('LANGUAGE_CLASSIFICATION_REASON', ''), 
            'LANGUAGE_CHECKED_AT': now_ast(),
            'TOKENS_IN':           row.get('ESTIMATED_INPUT_TOKENS', 0),
            'TOKENS_OUT':          2,
            'MODEL':                           'claude-haiku-4-5'
                                               if str(row.get('LANGUAGE_CLASSIFICATION_REASON', '')).startswith('Haiku')
                                               else 'claude-sonnet-5',
        } for row in language_check]),
        table_name='LANGUAGE_STAGING',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=True,
        auto_create_table=True,
    )

    s.sql(f"""
        UPDATE {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
        SET
            p.LANGUAGE_RESULT     = t.LANGUAGE_RESULT,
            p.LANGUAGE_CLASSIFICATION_REASON   = t.LANGUAGE_CLASSIFICATION_REASON, 
            p.LANGUAGE_CHECKED_AT = t.LANGUAGE_CHECKED_AT
        FROM {DB}.{PROCESSING_SCHEMA}.LANGUAGE_STAGING t
        WHERE p.DOC_ID         = t.DOC_ID
          AND p.PAGE_INDEX     = t.PAGE_INDEX
          AND p.LANGUAGE_RESULT IS NULL
    """).collect()

    info(f"Updated {len(language_check)} page(s) in DOCUMENTS_PAGES")

    s.sql(f"""
        INSERT INTO {DB}.{AUDIT_SCHEMA}.LLM_USAGE
            (DOC_ID, CHILD_DOC_ID, PIPELINE_STEP, MODEL_NAME, TOKENS_IN, TOKENS_OUT, CALLED_AT)
        SELECT
            DOC_ID,
            NULL,
            'LANGUAGE_CHECK',
            'MODEL',
            TOKENS_IN,
            TOKENS_OUT,
            LANGUAGE_CHECKED_AT
        FROM {DB}.{PROCESSING_SCHEMA}.LANGUAGE_STAGING
        WHERE TOKENS_IN > 0   -- exclude rule-based rows from LLM_USAGE
    """).collect()

    info(f"Wrote {len(ai_classified)} LLM_USAGE row(s)")

    s.sql(f"""
        DROP TABLE IF EXISTS {DB}.{PROCESSING_SCHEMA}.LANGUAGE_STAGING
    """).collect()

#  Summary
print(f"\n Language classification summary")
s.sql(f"""
    SELECT
        LANGUAGE_RESULT,
        COUNT(DISTINCT DOC_ID)  AS DOCS,
        COUNT(*)                AS PAGES
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE LANGUAGE_RESULT IS NOT NULL
    GROUP BY LANGUAGE_RESULT
    ORDER BY LANGUAGE_RESULT
""").show()

In [ ]:
# Pull NON_ENGLISH pages that need translation -Non english ONly

pages_to_translate = s.sql(f"""
    SELECT
        DOC_ID,
        PAGE_INDEX,
        PAGE_NUMBER,
        PAGE_CONTENT
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE LANGUAGE_RESULT          = 'NON_ENGLISH'
      AND PAGE_CONTENT_TRANSLATED  IS NULL
      AND PAGE_CONTENT             IS NOT NULL
      AND TRIM(PAGE_CONTENT)      <> ''
""").collect()

info(f"Pages needing translation: {len(pages_to_translate)}")

translation_results = []
translation_errors  = []

if not pages_to_translate:
    info("Nothing to translate.")


# Translate all NON_ENGLISH pages

if pages_to_translate:
    values_list = ', '.join(
        f"('{r['DOC_ID']}', {r['PAGE_INDEX']})"
        for r in pages_to_translate
    )

    try:
        raw_results = s.sql(f"""
            SELECT
                p.DOC_ID,
                p.PAGE_INDEX,
                p.PAGE_NUMBER,
                p.PAGE_CONTENT,
                LENGTH(p.PAGE_CONTENT) AS CONTENT_CHARS,
                LENGTH(?)              AS PROMPT_CHARS,
                AI_COMPLETE(
                    '{TRANSLATE_MODEL}',
                    REPLACE(?, '{{page_text}}', p.PAGE_CONTENT),
                    {{'max_tokens': 8192}}
                ) AS TRANSLATED
            FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
            JOIN (
                SELECT DISTINCT DOC_ID, PAGE_INDEX
                FROM (VALUES {values_list}) AS t(DOC_ID, PAGE_INDEX)
            ) AS to_translate
                ON  p.DOC_ID     = to_translate.DOC_ID
                AND p.PAGE_INDEX = to_translate.PAGE_INDEX
            WHERE p.PAGE_CONTENT IS NOT NULL
              AND TRIM(p.PAGE_CONTENT) <> ''
        """, params=[TRANSLATE_PROMPT, TRANSLATE_PROMPT]).collect()

        info(f"  {len(raw_results)} page(s) returned from Cortex")

    except Exception as e:
        error(f"  SQL translation failed: {e}")
        raw_results = []

    for row in raw_results:
        doc_id     = row['DOC_ID']
        page_index = row['PAGE_INDEX']
        raw        = row['TRANSLATED']

        try:
            if not raw or not raw.strip():
                raise ValueError("Empty response from AI_COMPLETE")

            translated = raw.strip()
            if translated.startswith('```'):
                parts      = translated.split('```')
                translated = parts[1].strip()
                if translated.lower().startswith(('text', 'md', 'markdown')):
                    translated = translated.split('\n', 1)[1].strip()

            translated = translated.replace('\\n', '\n')

            input_tokens  = (row['PROMPT_CHARS'] + row['CONTENT_CHARS']) // 4
            output_tokens = len(translated) // 4

            translation_results.append({
                'DOC_ID':                  doc_id,
                'PAGE_INDEX':              page_index,
                'PAGE_NUMBER':             row['PAGE_NUMBER'],
                'PAGE_CONTENT_TRANSLATED': translated,
                'INPUT_TOKENS':            input_tokens,
                'OUTPUT_TOKENS':           output_tokens,
            })

            info(f"  [OK] DOC_ID: {doc_id} PAGE: {page_index} — "
                 f"{row['CONTENT_CHARS']:,} chars translated")

        except Exception as e:
            translation_errors.append({
                'doc_id':     doc_id,
                'page_index': page_index,
                'error':      str(e),
            })
            error(f"  [FAIL] DOC_ID: {doc_id} PAGE: {page_index}: {e}")

In [ ]:
if translation_results:
    now = now_ast()

    s.write_pandas(
        pd.DataFrame([{
            'DOC_ID':                  r['DOC_ID'],
            'PAGE_INDEX':              r['PAGE_INDEX'],
            'PAGE_CONTENT_TRANSLATED': r['PAGE_CONTENT_TRANSLATED'],
            'TOKENS_IN':               r['INPUT_TOKENS'],
            'TOKENS_OUT':              r['OUTPUT_TOKENS'],
            'CALLED_AT':               now,
        } for r in translation_results]),
        table_name='TRANSLATION_STAGING',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=True,
        auto_create_table=True,
    )

    # Update DOCUMENTS_PAGES 
    s.sql(f"""
        UPDATE {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
        SET p.PAGE_CONTENT_TRANSLATED = t.PAGE_CONTENT_TRANSLATED
        FROM {DB}.{PROCESSING_SCHEMA}.TRANSLATION_STAGING t
        WHERE p.DOC_ID     = t.DOC_ID
          AND p.PAGE_INDEX = t.PAGE_INDEX
    """).collect()

    info(f"Updated {len(translation_results)} page(s) in DOCUMENTS_PAGES")

    # Insert LLM_USAGE from staging
    s.sql(f"""
        INSERT INTO {DB}.{AUDIT_SCHEMA}.LLM_USAGE
            (DOC_ID, CHILD_DOC_ID, PIPELINE_STEP, MODEL_NAME,
             TOKENS_IN, TOKENS_OUT, CALLED_AT)
        SELECT
            DOC_ID,
            NULL,
            'TRANSLATE',
            '{TRANSLATE_MODEL}',
            TOKENS_IN,
            TOKENS_OUT,
            CALLED_AT
        FROM {DB}.{PROCESSING_SCHEMA}.TRANSLATION_STAGING
    """).collect()

    info(f"Wrote {len(translation_results)} LLM_USAGE row(s)")

    # Drop staging
    s.sql(f"""
        DROP TABLE IF EXISTS {DB}.{PROCESSING_SCHEMA}.TRANSLATION_STAGING
    """).collect()

In [ ]:
# Summary
print(f"\n Translation summary")
print(f"  Translated successfully : {len(translation_results)}")
print(f"  Errors                  : {len(translation_errors)}")
if translation_results:
    print(f"  Total input tokens     : "
          f"{sum(r['INPUT_TOKENS'] for r in translation_results):,}")
    print(f"  Total output tokens    : "
          f"{sum(r['OUTPUT_TOKENS'] for r in translation_results):,}")

if translation_errors:
    print("\n  Failed pages:")
    for e in translation_errors:
        print(f"    DOC_ID: {e['doc_id']} PAGE: {e['page_index']}: {e['error']}")

print(f"\n Language result breakdown after translation ")
s.sql(f"""
    SELECT
        LANGUAGE_RESULT,
        COUNT(DISTINCT DOC_ID)              AS DOCS,
        COUNT(*)                            AS PAGES,
        COUNT(PAGE_CONTENT_TRANSLATED)      AS PAGES_WITH_TRANSLATION,
        COUNT(*) - COUNT(PAGE_CONTENT_TRANSLATED)
                                            AS PAGES_WITHOUT_TRANSLATION
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE LANGUAGE_RESULT IS NOT NULL
    GROUP BY LANGUAGE_RESULT
    ORDER BY LANGUAGE_RESULT
""").show()

print(f"\n Sample translated pages")
s.sql(f"""
    SELECT
        DOC_ID,
        PAGE_NUMBER,
        LANGUAGE_RESULT,
        LEFT(PAGE_CONTENT, 150)             AS ORIGINAL_PREVIEW,
        LEFT(PAGE_CONTENT_TRANSLATED, 150)  AS TRANSLATED_PREVIEW
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE LANGUAGE_RESULT         = 'NON_ENGLISH'
      AND PAGE_CONTENT_TRANSLATED IS NOT NULL
    LIMIT 5
""").show()